# QC statistics and plots

## Table of Contents:

* [0. Dependencies](#Dependencies)
* [1. Preparing the environment](#Preparing-the-environment)
* [2. Sequencing and mapping statistics](#Sequencing-and-mapping-statistics)
* [3. Library statistics](#Library-statistics)
* [4. Control samples](#Control-samples)
* [5. Sample correlation and PCA](#Sample-correlation-and-PCA)
* [6. Essentiality](#Essentiality)
* [7. Sample removal](#Sample-removal)
* [8. Scaled count and log fold changes](#Scaled-count-and-log-fold-changes)
* [9. Positional Bias](#Positional-Bias)

***

## Dependencies

Please see [INSTALL_README.md](../INSTALL_README.md) for the installation of software dependencies. This Notebook assumes that [R](https://cran.r-project.org/) and library dependencies have been installed and that the `RScript` command is available with access to the relevant data.

***

## Preparing the environment

Several paths are used as input to more than one script. To simplify the commands and improve usability, commonly used paths are stored as environment variables. Those variables are then used in the script arguments to shorten input and output data paths. 

This Notebook assumes that you have the following directory structure and files in place before running the commands: 

* **Top level directory** (`REPO_PATH`)
    * **DATA**
        * **pyCROQUET** (`PYCROQUET_OUTPUT_PATH`)
        * **QC** (`QC_PATH`)
        * **RDS** (`RDS_PATH`)
    * **METADATA**
        * *sample_annotations.tsv*

The scripts require several files to be present:

* `METADATA/sample_annotations.tsv` - sample metadata.

In [1]:
# Set the top level path for the repository
export REPO_PATH=$(dirname `pwd`)

# Show the repository path
echo "Repository path: ${REPO_PATH}"

# Check the repository path exists (don't need to check subdirectories)
if [ ! -d "${REPO_PATH}" ]; then
  echo "Top level directory path does not exist: ${REPO_PATH}"
fi

Repository path: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper


Once the top level directory has been set (this will likely be the path to your clone of the repository), we then set several other resusable paths as environment variables and create their directories if they don't exist. It should not be assumed this exist when you clone the repository as they may be present in the .gitignore files (e.g. output logs or large data files).

In [2]:
# Set environment variables for reusable paths
export QC_PATH="${REPO_PATH}/DATA/QC"
export RDS_PATH="${REPO_PATH}/DATA/RDS/QC"

# Show the paths (debug)
echo "QC output directory: ${QC_PATH}"
echo "RDS output directory: ${RDS_PATH}"

# Create the directories if they don't exist 
mkdir -p "${QC_PATH}"
mkdir -p "${RDS_PATH}"

QC output directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC
RDS output directory: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/RDS/QC


***

## Sequencing and mapping statistics

[pyCROQUET](https://github.com/cancerit/pycroquet) version 1.5.1 was used to quantify each CRAM using the redundant library (`paralog_library.pycroquet.tsv`) which was processed/formatted in [01_Data_Preparation.ipynb](01_Data_Preparation.ipynb). 

For each CRAM the following output files are generated:

* `[run]_[lane]#[tag].counts.tsv.gz` - frequency of each paired construct (excluded from repository due to size)
* `[run]_[lane]#[tag].cram` and `[run]_[lane]#[tag].cram.crai` - alignment of reads to guides (excluded from repository due to size)
* `[run]_[lane]#[tag].query_class.tsv.gz` - classification of each read pair (excluded from repository due to size)
* `[run]_[lane]#[tag].stats.json` - summary statistics

The script `SCRIPTS/QC/01_pycroquet_statistics.R` performs the following actions:

* Collates individual JSON statistics files from pyCROQUET into a data frame (`pycroquet_statistics`)
* Calculates the per-sample pyCROQUET statistics for selected metrics (`pycroquet_sample_statistics`)
* Plots the total number of read pairs per sample (`total_pairs_barplot` and `DATA/QC/total_read_pairs_per_sample.png`)
* Plots the total number of mapped and unmapped read pairs per sample (`mapped_read_pairs` and `DATA/QC/read_pairs_mapped_per_sample.png`)
* Plots the proportion of mapped and unmapped read pairs per sample (`prop_mapped_read_pairs` and `DATA/QC/proportion_of_read_pairs_mapped_per_sample.png`)

The R data objects can be loaded from `DATA/RDS/QC/pycroquet_statistics.Rdata`.

In [3]:
Rscript ${REPO_PATH}/SCRIPTS/QC/01_pycroquet_statistics.R \
    -d ${REPO_PATH} \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --pycroquet ${REPO_PATH}/DATA/pyCROQUET \
    --out ${REPO_PATH}/DATA/QC/all_samples

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading pyCROQUET JSON statistics from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/pyCROQUET
Calculating sample mapping statistics...
Plotting total number of read pairs per sample...
Total read pairs barplot written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/total_read_pairs_per_sample.png
Preparing mapping statistics...
Plotting mapping rates of read pairs per sample...
Number of reads mapping barplot written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/read_pairs_mapped_per_sample.png
Proportion of reads mapping barplot written to: /lustre/scratch124

*** 
## Total number of read pairs per sample

<img src="../DATA/QC/all_samples/total_read_pairs_per_sample.png" width=60% height=60%/>

## Total number of mapped and unmapped read pairs per sample
<img src="../DATA/QC/all_samples/read_pairs_mapped_per_sample.png" width=60% height=60%/> 

## Proportion of mapped and unmapped read pairs per sample
<img src="../DATA/QC/all_samples/proportion_of_read_pairs_mapped_per_sample.png" width=60% height=60%/>

***

## Library statistics

In [4]:
Rscript ${REPO_PATH}/SCRIPTS/QC/02_library_statistics.R \
    -d ${REPO_PATH} \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --counts ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.control_mean.tsv \
    --annotations 13 \
    --controls ${REPO_PATH}/METADATA/control_samples.txt \
    --out ${REPO_PATH}/DATA/QC/all_samples

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading count matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.filt.control_mean.tsv
Identifying control samples...
Read 9 items
Gathering count matrix...
Calculating library statistics...
Adding sample metadata to library statistics...
Plotting median counts per sample...
Median counts per sample barplot written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/median_normalised_counts_per_sample.png
Plotting low counts per sample...
Low counts per sample barplot written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/low_normal

***

## Number of low count guides

<img src="../DATA/QC/all_samples/low_normalised_counts_per_sample.png" width=60% height=60%/>

## Library coverage

<img src="../DATA/QC/all_samples/median_normalised_counts_per_sample.png" width=60% height=60%/> 

## Gini index

<img src="../DATA/QC/all_samples/gini_index_per_sample.png" width=60% height=60%/>  

***

## Control samples

In [5]:
Rscript ${REPO_PATH}/SCRIPTS/QC/03_control_samples.R \
    -d ${REPO_PATH} \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --counts ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.control_mean.tsv \
    --annotations 13 \
    --controls ${REPO_PATH}/METADATA/control_samples.txt \
    --out ${REPO_PATH}/DATA/QC/all_samples

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading count matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.filt.control_mean.tsv
Identifying control samples...
Read 9 items
Building pairwise correlation plot...
Correlation plot saved to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/control_correlation_normalised_counts.png
Building count distribution plot...
Count distribution plot saved to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/normalised_control_count_distributions.png
Building control essentiality plot...
Control essentiality plot saved to: /lustre/scratch124/cas

## Correlation of controls 

<img src="../DATA/QC/all_samples/control_correlation_normalised_counts.png" width=40% height=40%/> 

## Normalised count distribution (controls)

<img src="../DATA/QC/all_samples/normalised_control_count_distributions.png" width=60% height=60%/> 

## Essential dropout (control_mean)

<img src="../DATA/QC/all_samples/normalised_control_essential_distribution.png" width=40% height=40%/>

***

## Sample correlation and PCA

In [6]:
Rscript ${REPO_PATH}/SCRIPTS/QC/04_sample_correlation_and_pca.R \
    -d ${REPO_PATH} \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --counts ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.control_mean.tsv \
    --annotations 13 \
    --controls ${REPO_PATH}/METADATA/control_samples.txt \
    --out ${REPO_PATH}/DATA/QC/all_samples

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading count matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.filt.control_mean.tsv
Identifying control samples...
Read 9 items
Calculating sample count correlations...
Preparing sample count correlations...
Plotting sample count correlations...
Sample correlation plot saved to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/normalised_sample_count_correlation_boxplot.png
Principal component analysis...
Sample PCA plot saved to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/normalised_sample_count_pca.png
Saving R objects to file..

***

## Correlation within and between cell lines

<img src="../DATA/QC/all_samples/normalised_sample_count_correlation_boxplot.png" width=60% height=60%/>    

## Prinicpal component analysis (PCA)

<img src="../DATA/QC/all_samples/normalised_sample_count_pca.png" width=40% height=40%/> 

***

## Essentiality

In [7]:
Rscript ${REPO_PATH}/SCRIPTS/QC/05_sample_essentiality.R \
    -d ${REPO_PATH} \
    --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --counts ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.control_mean.tsv \
     --lfc ${REPO_PATH}/DATA/preprocessing/lfc_matrix.unscaled.tsv \
    --annotations 13 \
    --out ${REPO_PATH}/DATA/QC/all_samples

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading count matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.filt.control_mean.tsv
Reading fold change matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.unscaled.tsv
Gathering LFC matrix...
Plotting essential dropout: Lung NSCLC
Essential dropout plot written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_samples/normalised_sample_lfc_Lung_NSCLC.png
Plotting essential dropout: Melanoma
Essential dropout plot written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/all_s

## NNMD
<img src="../DATA/QC/all_samples/normalised_LFC_NNMD.png" width=60% height=60%/>

## LFC by guide type (Melanoma)
<img src="../DATA/QC/all_samples/normalised_sample_lfc_Melanoma.png" width=40% height=40%/> 

## LFC by guide type (Pancreas)
<img src="../DATA/QC/all_samples/normalised_sample_lfc_Pancreas.png" width=40% height=40%/>

## LFC by guide type (Lung NSCLC)
<img src="../DATA/QC/all_samples/normalised_sample_lfc_Lung_NSCLC.png" width=40% height=40%/>

*** 

## Sample removal

In [8]:
 Rscript ${REPO_PATH}/SCRIPTS/QC/06_remove_samples.R \
     -d ${REPO_PATH} \
     --mapping ${REPO_PATH}/METADATA/sample_annotations.tsv \
     --counts ${REPO_PATH}/DATA/preprocessing/count_matrix.filt.control_mean.tsv \
     --lfc ${REPO_PATH}/DATA/preprocessing/lfc_matrix.unscaled.tsv \
     --annotations 13 \
     --samples ${REPO_PATH}/DATA/QC/all_samples/samples_to_remove.txt \
     --plots_out ${REPO_PATH}/DATA/QC/with_samples_removed \
     --counts_out ${REPO_PATH}/DATA/preprocessing 

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading count matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.filt.control_mean.tsv
Reading LFC matrix from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.unscaled.tsv
Identifying user-defined samples to remove...
Read 10 items
Samples to remove: COLO 792 R1, SK-MEL-28 R2, COLO 792 R2, SK-MEL-5 R1, COLO 792 R3, WM3702 R2, WM3702 R3, WM3702 R1, COR-L23 R2, NCI-H1568 R3
Removing user-defined samples from counts...
Saving count matrix with user-defined samples removed...
Count matrix with user-defined samples removed written to: /lustre/scratch124/casm/team113/users/vo1/5429_paral

***

## PCA after sample removal

<img src="../DATA/QC/with_samples_removed/normalised_sample_count_pca.samples_removed.png" width=40% height=40%/>  

***

## Scaled count and log fold changes

Normalised log fold changes are scaled such that the median of the essential genes (`sgrna_group` = `Essential`) is 1 and the median of the safe-targeting guides (`sgrna_group` = `Safe-targeting control`) is 0.

Scaled count matrix is written to: `DATA/preprocessing/count_matrix.scaled.tsv`  
Scaled log fold change matrix is written to: `DATA/preprocessing/lfc_matrix.scaled.tsv` 

In [10]:
Rscript ${REPO_PATH}/SCRIPTS/QC/07_scale_lfcs.R \
    -d ${REPO_PATH} \
    -f ${REPO_PATH}/DATA/preprocessing/lfc_matrix.unscaled.samples_removed.tsv \
    -m ${REPO_PATH}/METADATA/sample_annotations.tsv \
    --annotations 13 \
    --plots_out ${REPO_PATH}/DATA/QC/with_samples_removed \
    --lfc_out ${REPO_PATH}/DATA/preprocessing 

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading sample annotations from: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/METADATA/sample_annotations.tsv
Reading LFC matrix: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.unscaled.samples_removed.tsv
Preparing LFC matrix...
Calculating replicate medians per cell line by library type...
Calculating safe and essential medians...
Combining median data frames...
Scaling LFCs...
Scaled LFC TSV written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.scaled.tsv
Building LFC distribution plot...
LFC distribution plot saved to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/QC/with_samples_removed/scaled_lfc_distributions.samples_removed.png
Building LFC dist

In [10]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/lfc_matrix.scaled.tsv | wc -l

23392


***

## LFC distribution pre- and post-scaling

<img src="../DATA/QC/with_samples_removed/scaled_lfc_distributions.samples_removed.png" width=60% height=60%/>    

## LFC distribution by guide type (violin)

<img src="../DATA/QC/with_samples_removed/scaled_lfc_guide_source_violin.samples_removed.png" width=60% height=60%/>    

## LFC distribution by guide type (barplot)

<img src="../DATA/QC/with_samples_removed/scaled_lfc_guide_source_barplot.samples_removed.png" width=60% height=60%/>    


In [11]:
Rscript ${REPO_PATH}/SCRIPTS/QC/08_convert_scaled_lfcs_to_scaled_counts.R \
    -d ${REPO_PATH} \
    -f ${REPO_PATH}/DATA/preprocessing/lfc_matrix.unscaled.samples_removed.tsv \
    -c ${REPO_PATH}/DATA/preprocessing/count_matrix.norm.samples_removed.tsv \
    --annotations 13 

System has not been booted with systemd as init system (PID 1). Can't operate.
Failed to connect to bus: Host is down
Reading LFC matrix: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/lfc_matrix.unscaled.samples_removed.tsv
Reading count matrix: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/preprocessing/count_matrix.norm.samples_removed.tsv
Gathering scaled LFC matrix...
Gathering unscaled count matrix...
Getting unscaled control counts...
Removing unwanted samples...
Adding unscaled control counts to unscaled count matrix...
Combining unscaled counts and scaled log fold changes...
Reverting scaled log fold changes to scaled counts...
Generating scaled count matrix...
Adding control_mean into scaled count matrix...
Scaled count matrix RDS written to: /lustre/scratch124/casm/team113/users/vo1/5429_paralog_sl_vicky_finalised_for_paper/DATA/RDS/preprocessing/count_matrix.scaled.rds
Scaled 

In [12]:
tail -n +2 ${REPO_PATH}/DATA/preprocessing/count_matrix.scaled.tsv | wc -l

23392
